# Fotogrametri — bilder → 3D-punktsky

**Kjør øverst: Rediger → Notatbokinnstillinger → Maskinvarakselerator → GPU (T4)**

Pipeline:
1. Last opp bilder (eller monter Google Drive)
2. Feature extraction (SIFT)
3. Sequential feature matching
4. Sparse SfM (Structure from Motion)
5. Dense MVS (Patch Match Stereo — bruker GPU)
6. Visualisering med Open3D

In [ ]:
# Installer avhengigheter (kjøres én gang per Colab-sesjon)
!pip install pycolmap open3d plotly -q

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU funnet:', result.stdout.strip())
else:
    print('ADVARSEL: Ingen GPU funnet. Dense reconstruction (MVS) krever CUDA-GPU.')
    print('Gå til Rediger → Notatbokinnstillinger → GPU')

## Last opp bilder

**Alternativ A** — Last opp direkte (maks ~50 bilder, forsvinner når sesjonen avsluttes)  
**Alternativ B** — Monter Google Drive (anbefalt for mange/store bilder, persistent)

In [ ]:
# === ALTERNATIV A: Last opp bilder direkte ===
import os
from google.colab import files

os.makedirs('/content/images', exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'/content/images/{name}', 'wb') as f:
        f.write(data)
print(f'Lastet opp {len(uploaded)} bilder til /content/images/')

In [ ]:
# === ALTERNATIV B: Monter Google Drive ===
# Legg bilder i «Min disk/fotogrametri/images/» i Drive, juster stien under om nødvendig
from google.colab import drive
drive.mount('/content/drive')

## Konfigurasjon

Juster `IMAGE_DIR` til mappen med bildene dine.

In [ ]:
from pathlib import Path

# --- Juster disse ---
IMAGE_DIR  = Path('/content/images')                        # Alternativ A (opplastet)
# IMAGE_DIR = Path('/content/drive/MyDrive/fotogrametri/images')  # Alternativ B (Drive)
OUTPUT_DIR = Path('/content/output')

# Matching: 'sequential' for bilder langs en strekning/runde (anbefalt)
#           'exhaustive' for vilkårlig rekkefølge (<30 bilder)
MATCHING = 'sequential'
OVERLAP  = 10   # Antall naboer å matche per bilde (sequential)
RUN_DENSE = True  # False = bare sparse (raskere, ingen GPU nødvendig)
# --------------------

images = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in
                {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'})
print(f'Fant {len(images)} bilder i {IMAGE_DIR}')
for p in images[:5]:
    print(' ', p.name)
if len(images) > 5:
    print(f'  ... og {len(images)-5} til')

## Pipeline

In [ ]:
import shutil
import pycolmap

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
database_path = OUTPUT_DIR / 'database.db'
sparse_dir    = OUTPUT_DIR / 'sparse'

if database_path.exists(): database_path.unlink()
if sparse_dir.exists():    shutil.rmtree(sparse_dir)

# --- Feature extraction ---
print('Steg 1/3 — Feature extraction (SIFT)...')
opts = pycolmap.FeatureExtractionOptions()
opts.sift.max_num_features = 8192
pycolmap.extract_features(database_path=database_path, image_path=IMAGE_DIR,
                           extraction_options=opts)
print('  Ferdig.')

In [ ]:
# --- Feature matching ---
print(f'Steg 2/3 — Feature matching ({MATCHING}, overlap={OVERLAP})...')
if MATCHING == 'sequential':
    pairing = pycolmap.SequentialPairingOptions()
    pairing.overlap = OVERLAP
    pycolmap.match_sequential(database_path=database_path, pairing_options=pairing)
else:
    pycolmap.match_exhaustive(database_path=database_path)
print('  Ferdig.')

In [ ]:
# --- Sparse SfM ---
print('Steg 3/3 — Sparse SfM reconstruction...')
sparse_dir.mkdir(parents=True, exist_ok=True)
models = pycolmap.incremental_mapping(
    database_path=database_path,
    image_path=IMAGE_DIR,
    output_path=sparse_dir,
)

if not models:
    raise RuntimeError(
        'SfM feilet: ingen rekonstruksjon produsert.\n'
        'Sjekk at bildene har tilstrekkelig overlapp og ble tatt langs en strekning.'
    )

best_id, best = max(models.items(), key=lambda kv: kv[1].num_reg_images())
best_model_dir = sparse_dir / str(best_id)

print(f'\nSparse rekonstruksjon:')
print(f'  Registrerte bilder : {best.num_reg_images()} / {len(images)}')
print(f'  3D-punkter         : {best.num_points3D()}')

In [ ]:
# --- Dense MVS (patch match stereo) ---
if not RUN_DENSE:
    print('RUN_DENSE=False — hopper over dense reconstruction.')
else:
    dense_dir = OUTPUT_DIR / 'dense'
    if dense_dir.exists(): shutil.rmtree(dense_dir)
    dense_dir.mkdir()

    print('Dense MVS steg 1/3 — Undistorter bilder...')
    pycolmap.undistort_images(
        output_path=dense_dir,
        input_path=best_model_dir,
        image_path=IMAGE_DIR,
    )

    print('Dense MVS steg 2/3 — Patch match stereo (GPU)...')
    pycolmap.patch_match_stereo(workspace_path=dense_dir)

    print('Dense MVS steg 3/3 — Fuser dybdekart...')
    fused_ply = dense_dir / 'fused.ply'
    pycolmap.stereo_fusion(
        output_path=fused_ply,
        workspace_path=dense_dir,
        output_type='ply',
    )
    print(f'\nFerdig! Dense punktsky: {fused_ply}')

## Visualisering

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Velg hvilken punktsky som vises
if RUN_DENSE and fused_ply.exists():
    import open3d as o3d
    pcd = o3d.io.read_point_cloud(str(fused_ply))
    label = 'Dense punktsky (MVS)'
else:
    # Fallback: sparse fra pycolmap
    rec  = pycolmap.Reconstruction(str(best_model_dir))
    pts  = rec.points3D
    xyz  = np.array([p.xyz for p in pts.values()])
    rgb  = np.array([p.color for p in pts.values()]) / 255.0
    import open3d as o3d
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    pcd.colors = o3d.utility.Vector3dVector(rgb)
    label = 'Sparse punktsky (SfM)'

# Subsample for rask visning (maks 200k punkter i Plotly)
MAX_DISPLAY = 200_000
xyz = np.asarray(pcd.points)
rgb = np.asarray(pcd.colors)
if len(xyz) > MAX_DISPLAY:
    idx = np.random.choice(len(xyz), MAX_DISPLAY, replace=False)
    xyz, rgb = xyz[idx], rgb[idx]

colors = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b in rgb]

fig = go.Figure(go.Scatter3d(
    x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
    mode='markers',
    marker=dict(size=1, color=colors, opacity=0.7),
))
fig.update_layout(
    title=f'{label} — {len(xyz):,} punkter',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z', aspectmode='data'),
    margin=dict(l=0, r=0, b=0, t=40),
    height=700,
)
fig.show()

In [ ]:
# Last ned PLY-filen til PC
from google.colab import files as colab_files

if RUN_DENSE and fused_ply.exists():
    colab_files.download(str(fused_ply))
else:
    sparse_ply = OUTPUT_DIR / 'sparse_pointcloud.ply'
    best.export_PLY(str(sparse_ply))
    colab_files.download(str(sparse_ply))